# Spark Architecture & Data Processing

## Overview
This notebook covers core Apache Spark concepts including driver-executor architecture, lazy evaluation, DAG execution, transformation types, and file format optimizations.

## Spark Architecture Components
1. **Driver**: The main controller process running the SparkSession. It builds the execution plan (DAG), divides it into stages and tasks, and manages execution across worker nodes.
2. **Cluster Manager**: Manages resources across physical nodes in the cluster (YARN, Kubernetes, Standalone).
3. **Executors**: Worker processes that execute tasks assigned by the Driver and store cached data in memory or disk.

## Key Concepts
- **Lazy Evaluation**: Spark records transformations in an execution graph (DAG) without computing results immediately. Execution is triggered only when an action is called.
- **DAG (Directed Acyclic Graph)**: Lineage graph of operations used by Catalyst Optimizer to prune unneeded columns, push filters down to storage, and recompute lost partitions on node failures.
- **Transformations vs. Actions**:
  - *Transformations* (`filter`, `select`, `withColumn`, `groupBy`): Lazy operations returning a new DataFrame.
  - *Actions* (`show`, `count`, `collect`, `write`): Eager operations that trigger job execution.

In [ ]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, when

# Set up Hadoop environment for Windows
hadoop_dir = os.path.abspath("../hadoop_win_custom").replace(os.sep, "/")
os.environ["HADOOP_HOME"] = hadoop_dir

spark = SparkSession.builder \
    .appName("SparkArchitectureAssignment") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.driver.extraJavaOptions", f"-Dhadoop.home.dir={hadoop_dir}") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

---
## Reading Data with Explicit Schema
Defining explicit schemas using `StructType` avoids schema inference overhead and ensures data types are strictly enforced.

In [ ]:
schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("transaction_date", StringType(), True)
])

dataset_path = "../data/dataset.csv"
raw_df = spark.read.schema(schema).option("header", "true").csv(dataset_path)

raw_df.printSchema()
raw_df.show()

---
## Cleaning and Feature Transformation
Handling null values, casting data types, and deriving additional business metrics (`total_price`, `is_expensive`).

In [ ]:
cleaned_df = raw_df.fillna({"amount": 0.0, "quantity": 1})

transformed_df = cleaned_df \
    .withColumnRenamed("amount", "total_amount") \
    .withColumnRenamed("city", "location") \
    .withColumn("quantity", col("quantity").cast("integer")) \
    .withColumn("total_price", col("total_amount") * col("quantity")) \
    .withColumn("is_expensive", when(col("total_amount") > 500, "Yes").otherwise("No"))

transformed_df.show()

---
## Narrow vs. Wide Transformations
- **Narrow Transformations** (`filter`, `select`): Each input partition maps to one output partition without data moving across the network.
- **Wide Transformations** (`groupBy`): Operations require redistributing data across cluster nodes (Shuffle operation).

In [ ]:
# Narrow Transformation
electronics_df = transformed_df \
    .filter((col("category") == "Electronics") & (col("total_amount") > 100)) \
    .select("transaction_id", "category", "total_amount", "location")

electronics_df.show()

# Wide Transformation (Triggers Shuffle)
category_sales = transformed_df \
    .groupBy("category") \
    .sum("total_amount") \
    .withColumnRenamed("sum(total_amount)", "category_total_sales")

category_sales.show()

---
## Performance Observations & Writing Output
1. **Predicate Pushdown & Projection Pruning**: Under `.explain()`, Spark scans only required columns and applies filter predicates at file reader level.
2. **CSV vs. Parquet**: Parquet uses columnar storage with compression, allowing column pruning and block skipping, whereas CSV is row-based.
3. **`show()` vs `collect()`**: `.show(n)` brings only n sample records to the driver, while `.collect()` pulls all data into driver memory which can cause OOM errors on large datasets.

In [ ]:
electronics_df.explain()

transformed_df.write.mode("overwrite").option("header", "true").csv("../output/results_csv")
transformed_df.write.mode("overwrite").parquet("../output/results_parquet")

spark.stop()